# Homework 08 - PyTorch Bridge

目标：把 micrograd 的动作逐个翻译到 PyTorch。PyTorch 需要装在本地 venv，不要装系统 Python。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

from course.checks import qa_check

## 完整例子 - micrograd 一步训练

In [ ]:
from micrograd.engine import Value

w = Value(2.0)
x = Value(3.0)
y = Value(7.0)
pred = w * x
loss = (pred - y) ** 2
loss.backward()
print('micrograd pred/loss/grad:', pred.data, loss.data, w.grad)
w.data -= 0.1 * w.grad
print('updated w:', w.data)

## 作业 1 - 先不 import torch，做概念映射

In [ ]:
student_mapping = None
# 期望：
# {
#   'Value': 'Tensor(requires_grad=True)',
#   'loss.backward()': 'loss.backward()',
#   'p.grad = 0': 'optimizer.zero_grad()',
#   'p.data -= lr*p.grad': 'optimizer.step()'
# }



qa_check('qa_check_08_mapping', globals(), student_mapping)

<details><summary>Show / Hide 本题提示</summary>

- 把 micrograd 词汇翻译过去：`Value -> Tensor(requires_grad=True)`，`backward -> loss.backward()`，更新由 optimizer 接手。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `mapping.get('Value')` 填 `'Tensor(requires_grad=True)'`。
- `mapping.get('loss.backward()')` 填 `'loss.backward()'`。
- `mapping.get('p.grad = 0')` 填 `'optimizer.zero_grad()'`。
- `mapping.get('p.data -= lr*p.grad')` 填 `'optimizer.step()'`。

</details>

## Modify - 同一件事，换学习率

micrograd 和 PyTorch 的更新本质一样：`new = old - lr * grad`。

In [ ]:
old_w = 2.0
grad = -6.0
# TODO: lr=0.01 时，新 w 是多少？
student_w_after_lr_001 = None



qa_check('qa_check_08_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 把 micrograd 词汇翻译过去：`Value -> Tensor(requires_grad=True)`，`backward -> loss.backward()`，更新由 optimizer 接手。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_w_after_lr_001` 约等于 `2.06`。

</details>

## 作业 2 - PyTorch 一步训练，可选实际运行

如果当前 kernel 没有 torch，这个 cell 会跳过运行，但概念题必须完成。

In [ ]:
try:
    import torch
except Exception as exc:
    torch = None
    print('torch 未安装。请用本地 venv：python3 -m venv .venv && .venv/bin/python -m pip install -r course/requirements-course.txt')

if torch is not None:
    w_t = torch.tensor(2.0, requires_grad=True)
    x_t = torch.tensor(3.0)
    y_t = torch.tensor(7.0)
    pred_t = w_t * x_t
    loss_t = (pred_t - y_t) ** 2
    loss_t.backward()
    print('torch pred/loss/grad:', pred_t.item(), loss_t.item(), w_t.grad.item())
    with torch.no_grad():
        w_t -= 0.1 * w_t.grad
    w_t.grad.zero_()
    print('updated w:', w_t.item(), 'grad after zero:', w_t.grad.item())
else:
    print('跳过 torch 运行；先把上面的映射写对。')

## Debug Lab - PyTorch 最常见两个坑

In [ ]:
# Debug 1：手动更新 tensor 参数时，为什么常放进 torch.no_grad()？
student_no_grad_reason = None

# Debug 2：为什么每轮要 optimizer.zero_grad()？
student_zero_grad_reason = None

# Debug 3：Tensor 比 Value 多出来的最大能力是什么？填 'vector/matrix/batch'。
student_tensor_extra_power = None



qa_check('qa_check_08_debug', globals())

<details><summary>Show / Hide 本题提示</summary>

- 把 micrograd 词汇翻译过去：`Value -> Tensor(requires_grad=True)`，`backward -> loss.backward()`，更新由 optimizer 接手。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- 答案需要满足：`isinstance(student_no_grad_reason, str) and ('图' in student_no_grad_reason or '记录' in student_no_grad_reason)`。
- 答案需要满足：`isinstance(student_zero_grad_reason, str) and ('累加' in student_zero_grad_reason or '清零' in student_zero_grad_reason)`。
- `student_tensor_extra_power` 填 `'vector/matrix/batch'`。

</details>

## 提交清单

- [ ] 概念映射通过
- [ ] 如果环境有 torch，实际跑过一步训练
- [ ] Debug Lab 通过
- [ ] 我能说出 PyTorch 封装了 micrograd 的哪些手写动作

<details><summary>Show / Hide 提示</summary>

先找完整例子的形状，再只改一个地方。填 TODO 前，先用小数字在纸上算一遍；测试失败时先判断错在数学、Python、计算图还是训练循环。

</details>

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>